In [2]:
import os
import json
from pathlib import Path
from semanticher.onto import OntologyDAG
from semanticher.data import load_table_list, load_tables, load_label_class

dag = OntologyDAG()
dag.build_dag()
df_table_list = load_table_list()
dict_tables   = load_tables()
df_labels     = load_label_class()

# =========================
# Base folders
# =========================
REPO_ROOT    = Path().resolve().parents[1]
RESULT_DIR   = REPO_ROOT / "results" / "main_edm" / "Result_edm"
EVAL_DIR     = REPO_ROOT / "results" / "main_edm" / "Eval_edm"

os.makedirs(EVAL_DIR, exist_ok=True)

# =========================
# Model aliases / experiments
# =========================
MODEL_ALIAS = {
    "deepseek-chat":    "dsk",
    "gemini-2.0-flash": "gem",
    "gpt-4.1-mini":     "gpt",
}

EXPERIMENTS = [
    ["deepseek-chat"],
    ["gemini-2.0-flash"],
    ["gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash"],
    ["deepseek-chat", "gpt-4.1-mini"],
    ["gemini-2.0-flash", "gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash", "gpt-4.1-mini"],
]


def build_experiment_tag(model_names):
    aliases = [MODEL_ALIAS[m] for m in model_names]
    prefix  = {1: "single", 2: "two", 3: "three"}[len(model_names)]
    return f"{prefix}_{'_'.join(aliases)}"

In [3]:
n_level1, n_level2 = 0, 0
for o in dag.edges.get(dag.root, []):
    n_level1 += 1
    print(dag.name_of_uri(o))
    for c in dag.edges.get(o, []):
        n_level2 += 1
        print("\t", dag.name_of_uri(c))
print(f"Number of level 1: {n_level1}")
print(f"Number of level 2: {n_level2}")

OccupantBehavior
PowerSystemResource
	 EquipmentContainer
	 PowerEquipment
PowerDeliveryUnit
District
KeyPerformanceIndicator
	 StrategicKPI
	 TacticalKPI
	 OperationalKPI
PerformanceGoal
KPICalculation
KPIEvaluatedObject
KPICalculationComponent
	 Assumption
	 Mathematical_model
	 UniversalConstant
	 Variable
	 Equation
	 DatumSource
Stakeholder
Unit_of_measure
	 Compound_unit
	 Singular_unit
KPIValue
ObservationValue
Observation
Schedule
FeatureOfInterest
ObservationProperty
	 EquipmentParameter
	 OccupancyParameter
	 EnergyParameter
	 EnvironmentalParameter
	 BuildingParameter
Slot
Ordered List
	 Route
Event
Location
GeographicalCoordinatesPoint
PhysicalObject
	 Building1
	 Occupant
	 Equipment
	 BuildingThing
Certificate
Observation
Observable Property
AC Meausrement
Property
	 Energy
	 Power
	 Power
	 Flow
	 State of charge
	 CO2 level
	 Current1
	 Voltage1
	 Tap position
	 Volume
	 AC Frequency
	 Alarm2
	 Current2
	 Day Type
	 Water Consumption
	 Gas Consumption
	 Heat Consumption

In [4]:
def path_level_f1_precision_recall(data):
    """
    Calculate micro and macro precision, recall, and F1
    """
    precisions = []
    recalls    = []
    f1s        = []
    total_tp   = 0
    total_fp   = 0
    total_fn   = 0
    for row in data:
        gt_set   = {tuple(p) for p in row["gt_paths"]}
        pred_set = {tuple(p) for p in row["pred_paths"]}
        tp = len(pred_set.intersection(gt_set))
        fp = len(pred_set - gt_set)
        fn = len(gt_set - pred_set)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
        total_tp += tp
        total_fp += fp
        total_fn += fn
    macro_precision = sum(precisions) / len(precisions) if precisions else 0.0
    macro_recall    = sum(recalls)    / len(recalls)    if recalls    else 0.0
    macro_f1        = sum(f1s)        / len(f1s)        if f1s        else 0.0
    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1        = 2 * (micro_precision * micro_recall) / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0.0
    print("Macro Precision:", macro_precision)
    print("Macro Recall:   ", macro_recall)
    print("Macro F1:       ", macro_f1)
    print("Micro Precision:", micro_precision)
    print("Micro Recall:   ", micro_recall)
    print("Micro F1:       ", micro_f1)
    return macro_precision, macro_recall, macro_f1, micro_precision, micro_recall, micro_f1


def flatten_list_to_set(paths):
    flat = set()
    for path in paths:
        for item in path:
            if isinstance(item, str):
                flat.add(item)
    return flat


def node_level_f1_precision_recall(data):
    """
    Calculate micro and macro precision, recall, and F1 at the node level
    """
    precisions = []
    recalls    = []
    f1s        = []
    total_tp   = 0
    total_fp   = 0
    total_fn   = 0
    for row in data:
        gt_set   = flatten_list_to_set(row["gt_paths"])
        pred_set = flatten_list_to_set(row["pred_paths"])
        tp = len(pred_set.intersection(gt_set))
        fp = len(pred_set - gt_set)
        fn = len(gt_set - pred_set)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
        total_tp += tp
        total_fp += fp
        total_fn += fn
    macro_precision = sum(precisions) / len(precisions) if precisions else 0.0
    macro_recall    = sum(recalls)    / len(recalls)    if recalls    else 0.0
    macro_f1        = sum(f1s)        / len(f1s)        if f1s        else 0.0
    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1        = 2 * (micro_precision * micro_recall) / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0.0
    print("Macro Precision:", macro_precision)
    print("Macro Recall:   ", macro_recall)
    print("Macro F1:       ", macro_f1)
    print("Micro Precision:", micro_precision)
    print("Micro Recall:   ", micro_recall)
    print("Micro F1:       ", micro_f1)
    return macro_precision, macro_recall, macro_f1, micro_precision, micro_recall, micro_f1

In [5]:
example_data = [
    {
        "table_id":   "1.csv",
        "table_name": "School building data.csv",
        "column_id":   0,
        "column_name": "Month",
        "pred_paths":  [["TemporalEntity", "Interval"]],
        "gt_paths":    [["TemporalEntity", "Interval"]]
    },
    {
        "table_id":   "1.csv",
        "table_name": "School building data.csv",
        "column_id":   1,
        "column_name": "Usage [kWh]",
        "pred_paths":  [],
        "gt_paths":    [["TemporalEntity", "Interval"]]
    },
    {
        "table_id":   "1.csv",
        "table_name": "School building data.csv",
        "column_id":   2,
        "column_name": "Cost (net in PLN)",
        "pred_paths":  [["ObservationValue"]],
        "gt_paths":    [["ObservationValue"]]
    },
    {
        "table_id":   "1.csv",
        "table_name": "School building data.csv",
        "column_id":   2,
        "column_name": "Cost (net in PLN)",
        "pred_paths":  [["ObservationValue", "Interval"]],
        "gt_paths":    [["ObservationValue", "TemporalEntity", "TemporalPosition"]]
    }
]

print("=== Path Level ===")
path_level_f1_precision_recall(example_data)
print()
print("=== Node Level ===")
node_level_f1_precision_recall(example_data)

=== Path Level ===
Macro Precision: 0.5
Macro Recall:    0.5
Macro F1:        0.5
Micro Precision: 0.6666666666666666
Micro Recall:    0.5
Micro F1:        0.5714285714285715

=== Node Level ===
Macro Precision: 0.625
Macro Recall:    0.5833333333333334
Macro F1:        0.6
Micro Precision: 0.8
Micro Recall:    0.5
Micro F1:        0.6153846153846154


(0.625, 0.5833333333333334, 0.6, 0.8, 0.5, 0.6153846153846154)

In [6]:
def run_single_eval(model_names):
    tag              = build_experiment_tag(model_names)
    result_file_path = RESULT_DIR / f"edm_{tag}.json"
    eval_json_path   = EVAL_DIR   / f"eval_edm_{tag}.json"
    eval_txt_path    = EVAL_DIR   / f"eval_edm_{tag}.txt"

    print("=" * 80)
    print(f"Evaluating: {tag}")
    print("=" * 80)

    with open(result_file_path) as f:
        result_pred = json.load(f)

    eval_json = []
    for r in result_pred:
        table_id    = r["table_id"]
        table_name  = r["table_name"]
        column_id   = r["column_id"]
        column_name = r["column_name"]

        label_row  = df_labels[
            (df_labels["table_id"]  == table_id) &
            (df_labels["column_id"] == column_id)
        ].iloc[0]

        label_c1l1 = label_row["class1_level1"]
        label_c1l2 = label_row["class1_level2"]
        label_c2l1 = label_row["class2_level1"]
        label_c2l2 = label_row["class2_level2"]

        pred_paths = [list(path) for path in r["paths"]]

        gt_paths = []
        if label_c1l1 != "-":
            gt_path = [label_c1l1]
            if label_c1l2 != "-":
                gt_path.append(label_c1l2)
            gt_paths.append(gt_path)
        if label_c2l1 != "-":
            gt_path = [label_c2l1]
            if label_c2l2 != "-":
                gt_path.append(label_c2l2)
            gt_paths.append(gt_path)

        eval_json.append({
            "table_id":    table_id,
            "table_name":  table_name,
            "column_id":   column_id,
            "column_name": column_name,
            "pred_paths":  pred_paths,
            "gt_paths":    gt_paths,
        })

    with open(eval_json_path, "w") as f:
        json.dump(eval_json, f, indent=2)

    print("+++++++++++++++++Path Level+++++++++++++++++")
    macro_precision, macro_recall, macro_f1, micro_precision, micro_recall, micro_f1 = path_level_f1_precision_recall(eval_json)
    print()
    print("+++++++++++++++++Node Level+++++++++++++++++")
    macro_precision_n, macro_recall_n, macro_f1_n, micro_precision_n, micro_recall_n, micro_f1_n = node_level_f1_precision_recall(eval_json)

    with open(eval_txt_path, "w") as f:
        f.write(f"Experiment: {tag}\n")
        f.write(f"Models:     {model_names}\n\n")
        f.write(f"+++++++++++++++++Path Level+++++++++++++++++\n")
        f.write(f"\tMacro Precision: {macro_precision}\n")
        f.write(f"\tMacro Recall:    {macro_recall}\n")
        f.write(f"\tMacro F1:        {macro_f1}\n")
        f.write(f"\tMicro Precision: {micro_precision}\n")
        f.write(f"\tMicro Recall:    {micro_recall}\n")
        f.write(f"\tMicro F1:        {micro_f1}\n\n")
        f.write(f"+++++++++++++++++Node Level+++++++++++++++++\n")
        f.write(f"\tMacro Precision: {macro_precision_n}\n")
        f.write(f"\tMacro Recall:    {macro_recall_n}\n")
        f.write(f"\tMacro F1:        {macro_f1_n}\n")
        f.write(f"\tMicro Precision: {micro_precision_n}\n")
        f.write(f"\tMicro Recall:    {micro_recall_n}\n")
        f.write(f"\tMicro F1:        {micro_f1_n}\n")

    print(f"\nSaved eval  -> {eval_json_path}")
    print(f"Saved score -> {eval_txt_path}")


def run_all_evals():
    for model_names in EXPERIMENTS:
        run_single_eval(model_names=model_names)

run_all_evals()

Evaluating: single_dsk
+++++++++++++++++Path Level+++++++++++++++++
Macro Precision: 0.25
Macro Recall:    0.125
Macro F1:        0.16666666666666666
Micro Precision: 0.25
Micro Recall:    0.2
Micro F1:        0.22222222222222224

+++++++++++++++++Node Level+++++++++++++++++
Macro Precision: 0.25
Macro Recall:    0.125
Macro F1:        0.16666666666666666
Micro Precision: 0.5
Micro Recall:    0.2
Micro F1:        0.28571428571428575

Saved eval  -> D:\RWTH\Github\sta-multi-llm-framework\results\main_edm\Eval_edm\eval_edm_single_dsk.json
Saved score -> D:\RWTH\Github\sta-multi-llm-framework\results\main_edm\Eval_edm\eval_edm_single_dsk.txt
Evaluating: single_gem
+++++++++++++++++Path Level+++++++++++++++++
Macro Precision: 0.25
Macro Recall:    0.125
Macro F1:        0.16666666666666666
Micro Precision: 0.25
Micro Recall:    0.2
Micro F1:        0.22222222222222224

+++++++++++++++++Node Level+++++++++++++++++
Macro Precision: 0.25
Macro Recall:    0.125
Macro F1:        0.1666666666666